# Phase 4: Demand Forecasting with Prophet
In this notebook, we visualize the synthesized time series and evaluate the Prophet forecasting models used by the RL simulator.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import joblib
from prophet import Prophet

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.simulator.demand_forecast import (
    prepare_prophet_df, 
    fit_prophet, 
    generate_forecast, 
    get_demand_for_datetime
)

os.makedirs("../visualization/exploration/timeseries", exist_ok=True)
plt.style.use('ggplot')

## 1. Load Synthesized Demand Data

In [ ]:
df_counts = pd.read_csv("../data/processed/demand_counts.csv")
df_counts["ds"] = pd.to_datetime(df_counts["ds"])
print(f"Total records: {len(df_counts)}")
df_counts.head()

## 2. Visualize Daily Demand per Location

In [ ]:
plt.figure(figsize=(15, 6))
sns.lineplot(data=df_counts, x="ds", y="y", hue="location")
plt.title("Synthesized Ride Demand (Count) per Location")
plt.ylabel("Number of Rides")
plt.xlabel("Timestamp")
plt.show()

## 3. Inspect Prophet Forecasts

In [ ]:
locations = df_counts["location"].unique()
for loc in locations:
    model_path = f"../models/prophet_{loc}.pkl"
    m = joblib.load(model_path)
    
    # Forecast for the next 28 days (112 slots)
    future = m.make_future_dataframe(periods=112, freq="6H")
    forecast = m.predict(future)
    
    # Plot
    fig = m.plot(forecast)
    plt.title(f"Forecast for {loc}")
    plt.savefig(f"../visualization/exploration/timeseries/prophet_forecast_{loc}.png")
    plt.show()
    
    # Components
    fig2 = m.plot_components(forecast)
    plt.savefig(f"../visualization/exploration/timeseries/prophet_components_{loc}.png")
    plt.show()